## Split Overmerged Authors — ORCID Conflict

Identifies `work_authors` rows where the work's own `raw_orcid` (from `openalex_works_base.authorships[].raw_orcid`) is non-null and **disagrees** with the ORCID attached to the assigned author profile. These are hard-evidence overmerges: the publisher-asserted ORCID on the work is the most trustworthy attribution signal we have, and it trumps the clustering guess that put the row on an ORCID-bearing profile of the same name.

**Complementary to Phase 1 / Phase 2 / Middle-Name.** Those phases already use `raw_orcid != profile.orcid` as a *keep* predicate when the name geometry fires (disjoint last, disjoint first, disjoint middle). The gap they leave: same-first + same-last + same-middle rows where only the ORCID disagrees. This notebook closes that gap with no name-geometry predicate — pure ORCID conflict.

**Evidence:** Across 364.7M `work_authors` rows attached to an ORCID-bearing profile, only **74,465** carry a conflicting raw_orcid (0.020%). Of those: ~57k same-first/same-last (Phase 5 scope), ~12k different-first/same-last (Phase 2 scope but excluded by length/CJK/cluster filters), ~5k different-last (Phase 1 scope likewise excluded). 25-row spot-check of the 74k: ~92–96% precision. Dominant residual risk class: same real person registered for two ORCIDs (rare, concentrated in HEP/CERN authors).

**Safety filters:** none beyond `raw_orcid` and `profile.orcid` both non-null/non-empty. No cluster cap — one hard-mismatch row is enough. No CJK exclusion — ORCID is script-agnostic.

**Rollback plan:** MatchAuthors natural loop. A split row that was actually correct (dual-ORCID same person) will land in `STILL_UNMATCHED` and either re-attach to a different profile with matching ORCID, or sit until a later signal resolves it. No explicit rollback cell — this rule's false-positive class (dual ORCID registration) is better handled by author-profile merge downstream than by restoring the pre-split state.

### Cell 1: Build candidates table

In [ ]:
CREATE OR REPLACE TABLE openalex.authors.overmerge_split_candidates_orcid_conflict AS
WITH work_orcids AS (
  SELECT wb.id AS work_id, pos AS author_sequence, authorship.raw_orcid
  FROM openalex.works.openalex_works_base wb
  LATERAL VIEW POSEXPLODE(wb.authorships) t AS pos, authorship
  WHERE authorship.raw_orcid IS NOT NULL AND authorship.raw_orcid != ''
),
profile_orcids AS (
  SELECT author_id, orcid, first, last, full_name, works_count
  FROM openalex.authors.authors_for_matching
  WHERE orcid IS NOT NULL AND orcid != ''
),
conflicts AS (
  SELECT wa.work_id, wa.author_sequence, wa.author_id, wa.raw_author_name,
    po.orcid AS profile_orcid, wo.raw_orcid AS work_raw_orcid,
    po.first AS author_first, po.last AS author_last, po.full_name AS author_full_name,
    po.works_count
  FROM openalex.works.work_authors wa
  JOIN profile_orcids po ON wa.author_id = po.author_id
  JOIN work_orcids wo ON wo.work_id = wa.work_id AND wo.author_sequence = wa.author_sequence
  WHERE wo.raw_orcid != po.orcid
)
SELECT c.*,
  COUNT(*) OVER (PARTITION BY c.author_id) AS cluster_works
FROM conflicts c

### Cell 2: Validation statistics

In [ ]:
SELECT
  COUNT(*)                                                               AS total_candidates,
  COUNT(DISTINCT author_id)                                              AS distinct_authors,
  COUNT(DISTINCT work_id)                                                AS distinct_works,
  COUNT(DISTINCT CASE WHEN cluster_works  = 1              THEN author_id END) AS profiles_with_1,
  COUNT(DISTINCT CASE WHEN cluster_works BETWEEN 2 AND 5   THEN author_id END) AS profiles_with_2_to_5,
  COUNT(DISTINCT CASE WHEN cluster_works BETWEEN 6 AND 20  THEN author_id END) AS profiles_with_6_to_20,
  COUNT(DISTINCT CASE WHEN cluster_works > 20              THEN author_id END) AS profiles_gt_20,
  PERCENTILE_APPROX(works_count, ARRAY(0.25, 0.5, 0.75, 0.95))           AS author_works_pctiles
FROM openalex.authors.overmerge_split_candidates_orcid_conflict

### Cell 3: Spot-check sample (manual review)

In [ ]:
SELECT work_id, author_id, raw_author_name, author_full_name,
  author_first, author_last, profile_orcid, work_raw_orcid,
  cluster_works, works_count AS author_works_count
FROM openalex.authors.overmerge_split_candidates_orcid_conflict
ORDER BY RAND()
LIMIT 50

### Cell 4: Create audit log (for rollback / analysis)

In [ ]:
CREATE OR REPLACE TABLE openalex.authors.overmerge_split_log_orcid_conflict AS
SELECT work_id, author_sequence, author_id AS previous_author_id,
  raw_author_name, profile_orcid, work_raw_orcid,
  author_full_name, current_timestamp() AS split_timestamp
FROM openalex.authors.overmerge_split_candidates_orcid_conflict

### Cell 5: Execute the split

In [ ]:
MERGE INTO openalex.works.work_authors AS target
USING (
  SELECT DISTINCT work_id, author_sequence
  FROM openalex.authors.overmerge_split_candidates_orcid_conflict
) AS source
ON target.work_id = source.work_id
   AND target.author_sequence = source.author_sequence
WHEN MATCHED THEN
  UPDATE SET
    target.author_id = NULL,
    target.updated_at = current_timestamp()

### Cell 6: Queue split works for reprocessing by UpdateWorkAuthorships

In [ ]:
INSERT INTO openalex.institutions.curated_work_ids_pending_sync
SELECT DISTINCT work_id, NULL AS curated_ras, current_timestamp() AS added_datetime
FROM openalex.authors.overmerge_split_candidates_orcid_conflict

### Cell 7: Post-split verification

In [ ]:
SELECT
  COUNT(*) AS nulled_records,
  COUNT(DISTINCT c.work_id) AS works_affected
FROM openalex.works.work_authors wa
JOIN openalex.authors.overmerge_split_log_orcid_conflict c
  ON wa.work_id = c.work_id AND wa.author_sequence = c.author_sequence
WHERE wa.author_id IS NULL

### Cell 8: Outcome tracking (run after MatchAuthors re-processes)

In [ ]:
SELECT
  CASE
    WHEN wa.author_id IS NULL                       THEN 'STILL_UNMATCHED'
    WHEN wa.author_id = log.previous_author_id      THEN 'RE_MATCHED_SAME'
    ELSE                                                 'RE_MATCHED_NEW'
  END AS outcome,
  COUNT(*) AS cnt
FROM openalex.works.work_authors wa
JOIN openalex.authors.overmerge_split_log_orcid_conflict log
  ON wa.work_id = log.work_id AND wa.author_sequence = log.author_sequence
GROUP BY 1